# Overview

This notebook corresponds to makemore part 3.


In [ ]:
import random

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Load Data and Build Datasets

Removed checks and EDA (covered in part 2 notebook)


In [ ]:
with open("names.txt", "r") as file:
    names = file.read().splitlines()

In [ ]:
chars = ["."] + sorted(set("".join(names)))
s_to_i = {s: i for i, s in enumerate(chars)}
i_to_s = {i: s for i, s in enumerate(chars)}

vocab_size = len(chars)


def build_dataset(names, block_size=3):
    X, y = [], []
    for name in names:
        context = [0] * block_size
        name += "."
        for ch in name:
            idx = s_to_i[ch]
            X.append(context)
            y.append(idx)
            context = context[1:] + [idx]

    return torch.tensor(X), torch.tensor(y)

In [ ]:
random.seed(42)
random.shuffle(names)

p1, p2 = int(len(names) * 0.8), int(len(names) * 0.9)

block_size = 8

X_train, y_train = build_dataset(names[:p1], block_size)
X_val, y_val = build_dataset(names[p1:p2], block_size)
X_test, y_test = build_dataset(names[p2:], block_size)

In [ ]:
for x, y in zip(X_train[:10], y_train[:10]):
    print("".join(i_to_s[idx.item()] for idx in x), "-->", i_to_s[y.item()])

# Build MLP


In [ ]:
# fmt: off
g = torch.Generator().manual_seed(2147483647)


class Linear:
    def __init__(self, in_features, out_features, bias=True):
        self.weights = torch.randn((in_features, out_features), generator=g) / in_features**0.5
        self.bias = torch.zeros(out_features) if bias else None

    def __call__(self, x):
        result = x @ self.weights
        if self.bias is not None:
            result += self.bias

        self.out = result
        return self.out

    def parameters(self):
        return [self.weights] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
    def __init__(self, num_features, eps=1e-5, momentum=1e-3):
        self.eps = eps
        # Changed default momentum to be better for this context
        self.momentum = momentum

        self.gamma = torch.ones(num_features)
        self.beta = torch.zeros(num_features)

        self.running_mean = torch.zeros(num_features)
        self.running_var = torch.ones(num_features)

        self.training = True

    def __call__(self, x):
        if self.training:
            if x.ndim == 2:
                dim = 0
            elif x.ndim == 3:
                dim = (0, 1)

            x_mean = x.mean(dim)
            x_var = x.var(dim)

            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * x_mean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * x_var

            xhat = (x - x_mean) / torch.sqrt(x_var + self.eps)
            self.out = self.gamma * xhat + self.beta
            return self.out

        else:
            xhat = (x - self.running_mean) / torch.sqrt(self.running_var + self.eps)
            self.out = self.gamma * xhat + self.beta
            return self.out

    def parameters(self):
        return [self.gamma, self.beta]


class Tanh:
    def __init__(self):
        return

    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []

class Embedding:
    def __init__(self, num_embeddings, embedding_dim):
        self.weights = torch.randn((num_embeddings, embedding_dim))

    def __call__(self, x):
        self.out = self.weights[x]
        return self.out

    def parameters(self):
        return [self.weights]

class Flatten:
    def __init__(self):
        return

    def __call__(self, x):
        self.out = x.reshape(x.shape[0], -1)
        return self.out

    def parameters(self):
        return []

class FlattenConsec:
    def __init__(self, n):
        self.n = n

    def __call__(self, x):
        # For position in batch, sequence length, and channel
        B, L, C = x.shape
        x =  x.reshape(B, L // self.n, C * self.n)
        if x.shape[1] == 1:
            x = x.squeeze(1)

        self.out = x
        return self.out

    def parameters(self):
        return []

class Sequential:
    def __init__(self, layers):
        self.layers = layers

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)

        self.out = x
        return self.out

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


In [ ]:
torch.manual_seed(42)

In [ ]:
# Hyperparams set to values in video
emb_dim = 24
hidden_size = 128

# fmt: off
model = Sequential([
    # Embedding matrix
    Embedding(vocab_size, emb_dim), 
    # Dilated causal convolution
    FlattenConsec(2), Linear(emb_dim * 2, hidden_size, bias=False), BatchNorm1d(hidden_size), Tanh(),
    # Dilated causal convolution
    FlattenConsec(2), Linear(hidden_size * 2, hidden_size, bias=False), BatchNorm1d(hidden_size), Tanh(),
    # Dilated causal convolution
    FlattenConsec(2), Linear(hidden_size * 2, hidden_size, bias=False), BatchNorm1d(hidden_size), Tanh(),
    # Output layer
    Linear(hidden_size, vocab_size),
])
# fmt: on

with torch.no_grad():
    # Make predictions less confident at the start
    model.layers[-1].weights *= 0.1

params = model.parameters()
for p in params:
    p.requires_grad = True

print(f"Total params: {sum(p.numel() for p in params)}")

In [ ]:
idx = torch.randint(0, X_train.shape[0], (4,))
Xb, yb = X_train[idx], y_train[idx]
logits = model(Xb)
print(Xb.shape)
Xb

In [ ]:
for layer in model.layers:
    print(layer.__class__.__name__, ":", tuple(layer.out.shape))

In [ ]:
logits.shape

In [ ]:
model.layers[3].running_mean.shape

# Train and Evaluate MLP


In [ ]:
batch_size = 32
num_steps = 200000
lrs = [0.02, 0.2, 0.02, 0.002]
milestones = [2000, 120000, 150000]

# Reset generator for reproducibility
g = torch.Generator().manual_seed(42)

for layer in model.layers:
    if isinstance(layer, BatchNorm1d):
        layer.training = True

losses = []
j = 0
for i in range(num_steps):
    # Get mini-batch
    idxs = torch.randint(0, X_train.shape[0], (batch_size,), generator=g)
    X_batch, y_batch = X_train[idxs], y_train[idxs]

    # Forward pass
    x = X_batch
    y_pred = model(x)

    # Compute loss
    loss = F.cross_entropy(y_pred, y_batch)

    # Backward pass

    # Get gradients
    for p in params:
        p.grad = None
    loss.backward()

    # Set LR
    if j < len(milestones) and i >= milestones[j]:
        j += 1
    lr = lrs[j]

    # GD step
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad

    losses.append(loss.item())

    if i % 10000 == 0:
        print(f"Epoch: {i}/{num_steps}, loss: {loss}")

    # if i >= 100:
    #     break

Change in mean loss over chunks of 200 time steps


In [ ]:
plt.plot(torch.tensor(losses).reshape(-1, 200).mean(1))

In [ ]:
def evaluate(X, y):
    for layer in model.layers:
        if isinstance(layer, BatchNorm1d):
            layer.training = False

    y_pred = model(X)

    return F.cross_entropy(y_pred, y).item()

In [ ]:
train_loss = evaluate(X_train, y_train)
val_loss = evaluate(X_val, y_val)

print(f"Train loss: {train_loss}")
print(f"Val loss: {val_loss}")

Batch size 256:

- Train loss: 1.5463432
- Val loss: 2.1837373

Batch size 32:

- Train loss: 1.8140029
- Val loss: 1.9907788

I believe there was a research paper that found batch size doesn't inherently mean worse generalization. The paper found then when proper hyperparameter tuning was done, batch size basically didn't affect generalization at all. However, the results above do support the idea that when extensive tuning isn't done, higher batch size does negaively impact generalization (i.e., a small batch size acts as a regularizer).


# Inference


In [ ]:
for _ in range(5):
    context = [0] * block_size
    out = ""
    while True:
        for layer in model.layers:
            if isinstance(layer, BatchNorm1d):
                layer.training = False

        y_pred = model(torch.tensor([context]))
        y_probs = F.softmax(y_pred, dim=1)

        idx = torch.multinomial(y_probs, num_samples=1).item()
        if idx == 0:
            break
        context = context[1:] + [idx]
        out += i_to_s[idx]

    print(out)
